# EXP027: DQN on real response data

実回答データを用いる DQN notebook です。ネットワーク、ε-greedy、replay buffer、TD ターゲットは元論文添付の `Train and test DQN on the simulated banks.py` と EXP027 のシミュレーション版を基にしています。

研究条件：

- `Config.dataset` で `LNIRT_CredentialForm1` または `ShinyItemAnalysis_dataMedical` を選択します。
- 応答は3PLから生成せず、選択項目に対応する実回答CSVの値を使用します。
- 報酬は、CSV から読み込んだ `true theta` （LNIRT 等で推定された参照能力値 $\theta$）における Fisher 項目情報量です。EXP027 のシミュレーション版と同様に「真の θ」を報酬計算に用いる方針です。
- training CSVをseed固定で学習・検証に分割し、testing CSVは最終評価だけに使用します。
- 能力推定は `Config.estimation_method` で `MLE` または `EAP` を選択します。EAPは62点求積、一様事前分布 $U(-4,4)$ です。
- CSVの `true theta` は全項目回答から得た参照能力値であり、訓練時の報酬計算と検証・評価指標の計算の両方に使用します。

In [37]:
# -*- coding: utf-8 -*-
import copy
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Tuple, cast

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from scipy.optimize import minimize_scalar

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")


def find_project_root():
    candidates = []
    if "__file__" in globals():
        script_dir = Path(__file__).resolve().parent
        candidates.extend([script_dir, *script_dir.parents])

    cwd = Path.cwd().resolve()
    candidates.extend(
        [
            cwd,
            *cwd.parents,
            cwd / "Grad_Research",
            cwd
            / "Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History",
            Path("/content/Grad_Research"),
            Path(
                "/content/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History"
            ),
            Path("/content/drive/MyDrive/Grad_Research"),
            Path(
                "/content/drive/MyDrive/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History"
            ),
            Path("/content/drive/MyDrive/Colab Notebooks/Grad_Research"),
            Path(
                "/content/drive/MyDrive/Colab Notebooks/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History"
            ),
        ]
    )

    for root in candidates:
        if (root / "data").is_dir():
            return root

    try:
        from google.colab import drive

        drive.mount("/content/drive")
    except Exception:
        pass

    for root in candidates:
        if (root / "data").is_dir():
            return root

    raise FileNotFoundError(
        "Could not find the project root. "
        "In Colab, place the repository under /content or MyDrive."
    )


ROOT = find_project_root()
EXP027_DIR = ROOT / "EXP027"
MODEL_DIR = EXP027_DIR / "models"
RESULTS_DIR = EXP027_DIR / "results"

print(f"Device      : {device}")
print(f"Project root: {ROOT}")
print(f"Model dir   : {MODEL_DIR}")
print(f"Results dir : {RESULTS_DIR}")

Device      : mps
Project root: /Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History
Model dir   : /Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History/EXP027/models
Results dir : /Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History/EXP027/results


In [38]:
@dataclass
class Config:
    # Network
    input_size: int = 1
    first_hidden: int = 50
    second_hidden: int = 30
    dropout_rate: float = 0.0

    # Training
    test_length: int = 40
    gamma: float = 0.1
    memory_capacity: int = 1000
    epsilon: float = 0.1
    batch_size: int = 128
    q_network_iteration: int = 40
    learning_rate: float = 1e-3
    training_size: int = 0  # 0: use all rows remaining after validation split
    validation_size: int = 200
    validation_interval: int = 50

    # Ability estimation: 'MLE' | 'EAP'
    estimation_method: str = "MLE"
    n_quad: int = 62
    prior_low: float = -4.0
    prior_high: float = 4.0

    # Real response data
    dataset: str = "LNIRT_CredentialForm1"
    n_items: int = 0  # 0: use all items
    testing_size: int = 0  # 0: use all testing rows

    # Reproducibility
    seed: int = 20260430


@dataclass
class RealData:
    dataset: str
    item_bank: np.ndarray
    train_resp: np.ndarray
    train_theta: np.ndarray
    valid_resp: np.ndarray
    valid_theta: np.ndarray
    test_resp: np.ndarray
    test_theta: np.ndarray

In [39]:
def FI(item_para, theta, D=1):
    a = item_para[:, 0]
    b = item_para[:, 1]
    c = item_para[:, 2]
    return (
        D**2
        * a**2
        * (1 - c)
        / (c + np.exp(D * a * (theta - b)))
        / (1 + np.exp(-D * a * (theta - b))) ** 2
    )


def MLE(item_paras, resp, D=1):
    a = item_paras[:, 0]
    b = item_paras[:, 1]
    c = item_paras[:, 2]

    def mins_likelihood(x):
        logl = 0
        for i in range(len(resp)):
            p = (1 - c[i]) / (1 + np.exp(-D * a[i] * (x - b[i]))) + c[i]
            p = np.clip(p, 1e-10, 1 - 1e-10)
            logl -= resp[i] * np.log(p) + (1 - resp[i]) * np.log(1 - p)
        return logl

    result = cast(
        Any, minimize_scalar(mins_likelihood, bounds=(-4, 4), method="bounded")
    )
    return np.array(result.x).reshape(
        1,
    )


def MLE_TEST(item_paras, resp, D=1):
    def mins_likelihood(x):
        logl = 0
        for i in range(resp_i.shape[0]):
            p = (1 - c[i]) / (1 + np.exp(-D * a[i] * (x - b[i]))) + c[i]
            p = np.clip(p, 1e-10, 1 - 1e-10)
            logl -= resp_i[i] * np.log(p) + (1 - resp_i[i]) * np.log(1 - p)
        return logl

    theta = np.zeros(resp.shape[1])
    for i in range(resp.shape[1]):
        resp_i = resp[:, i]
        a = item_paras[:, i, 0]
        b = item_paras[:, i, 1]
        c = item_paras[:, i, 2]
        result = cast(
            Any, minimize_scalar(mins_likelihood, bounds=(-4, 4), method="bounded")
        )
        theta[i] = result.x
    return np.expand_dims(theta, axis=0)


def EAP_quadrature(
    item_paras: np.ndarray,
    resp: np.ndarray,
    n_quad: int = 62,
    prior_low: float = -4.0,
    prior_high: float = 4.0,
    D: float = 1.0,
) -> Tuple[float, float]:
    theta_grid = np.linspace(prior_low, prior_high, n_quad)
    log_prior = np.zeros(n_quad)

    a = item_paras[:, 0]
    b = item_paras[:, 1]
    c = item_paras[:, 2]
    p = c[:, None] + (1 - c[:, None]) / (
        1 + np.exp(-D * a[:, None] * (theta_grid[None, :] - b[:, None]))
    )
    p = np.clip(p, 1e-10, 1 - 1e-10)
    log_lik = np.sum(
        resp[:, None] * np.log(p) + (1 - resp[:, None]) * np.log(1 - p),
        axis=0,
    )

    log_post = log_lik + log_prior
    log_post -= log_post.max()
    post = np.exp(log_post)
    post /= post.sum()

    eap_mean = float(np.sum(theta_grid * post))
    eap_var = float(np.sum((theta_grid - eap_mean) ** 2 * post))
    return eap_mean, eap_var


def get_estimation_method(cfg):
    method = cfg.estimation_method.upper()
    if method not in {"MLE", "EAP"}:
        raise ValueError(
            f"Unsupported estimation_method: {cfg.estimation_method!r}. "
            "Use 'MLE' or 'EAP'."
        )
    return method


# Clip ability estimates to the MLE optimisation bounds so the all-correct /
# all-incorrect fallback cannot drift past the estimation range toward the
# extreme (winsorized) item difficulties (e.g. b_min ~ -6). EAP is already
# bounded to [prior_low, prior_high], so only the MLE / fallback paths clip.
THETA_MIN, THETA_MAX = -4.0, 4.0


def estimate_theta_single(cfg, item_bank, administered_items, responses, current_theta):
    if get_estimation_method(cfg) == "EAP":
        theta_hat, _ = EAP_quadrature(
            administered_items,
            responses,
            n_quad=cfg.n_quad,
            prior_low=cfg.prior_low,
            prior_high=cfg.prior_high,
        )
        return theta_hat

    if len(np.unique(responses)) == 1:
        if responses[-1] == 1:
            theta_hat = current_theta + (item_bank[:, 1].max() - current_theta) / 2
        else:
            theta_hat = current_theta - (current_theta - item_bank[:, 1].min()) / 2
    else:
        theta_hat = float(MLE(administered_items, responses)[0])
    return float(np.clip(theta_hat, THETA_MIN, THETA_MAX))


def estimate_theta_batch(cfg, item_bank, item_ids, responses, current_theta):
    testing_size = responses.shape[1]
    theta_hat = np.zeros(testing_size)

    if get_estimation_method(cfg) == "EAP":
        for subject in range(testing_size):
            theta_hat[subject], _ = EAP_quadrature(
                item_bank[item_ids[:, subject]],
                responses[:, subject],
                n_quad=cfg.n_quad,
                prior_low=cfg.prior_low,
                prior_high=cfg.prior_high,
            )
        return theta_hat

    idx_full = np.sum(responses, axis=0) == responses.shape[0]
    idx_zero = np.sum(responses, axis=0) == 0
    idx_norm = ~(idx_full | idx_zero)
    theta_hat[idx_full] = (
        current_theta[idx_full] + (item_bank[:, 1].max() - current_theta[idx_full]) / 2
    )
    theta_hat[idx_zero] = (
        current_theta[idx_zero] - (current_theta[idx_zero] - item_bank[:, 1].min()) / 2
    )
    if np.any(idx_norm):
        theta_hat[idx_norm] = np.squeeze(
            MLE_TEST(
                item_bank[item_ids[:, idx_norm]],
                responses[:, idx_norm],
            )
        )
    return np.clip(theta_hat, THETA_MIN, THETA_MAX)


def estimation_label(cfg):
    return "EAP_unif" if get_estimation_method(cfg) == "EAP" else "MLE"

In [40]:
class Net(nn.Module):
    def __init__(
        self, input_size, first_hidden, second_hidden, action_space, dropout_rate
    ):
        super().__init__()
        self.fc1 = nn.Linear(input_size, first_hidden)
        self.fc2 = nn.Linear(first_hidden, second_hidden)
        self.out = nn.Linear(second_hidden, action_space)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        x = F.relu(self.dropout(self.fc1(x)))
        x = F.relu(self.dropout(self.fc2(x)))
        return self.out(x)

    def initialize(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.kaiming_normal_(module.weight)


def Choose_Action(item_id, state, epsilon):
    if np.random.rand() >= epsilon:
        state_t = torch.unsqueeze(torch.FloatTensor(state), 0).to(device)
        action_value = eval_net(state_t)
        if item_id.size > 0:
            item_id_t = torch.from_numpy(item_id).to(device).long()
            action_value[:, item_id_t] = -torch.inf
        return torch.max(action_value, -1)[1].cpu().numpy()

    available = np.delete(np.arange(action_space), item_id)
    return (
        np.random.choice(available)
        .astype("int64")
        .reshape(
            1,
        )
    )


def Choose_Action_Test(item_id, state):
    state_t = torch.FloatTensor(state.T).to(device)
    action_value = eval_net(state_t).detach().cpu().numpy()
    if item_id.shape[0] > 0:
        subject_indices = np.arange(item_id.shape[1])[:, None]
        action_value[subject_indices, item_id.T] = -np.inf
    return action_value.argmax(axis=1)


def Apply_Positive_Constraint(model, min_value=0.0):
    for param in model.parameters():
        param.data = torch.clamp(param.data, min=min_value)

In [41]:
DATASET_DIRS = {
    "LNIRT_CredentialForm1": ROOT / "data" / "LNIRT_CredentialForm1",
    "ShinyItemAnalysis_dataMedical": (ROOT / "data" / "ShinyItemAnalysis_dataMedical"),
}


def set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def _read_theta(path):
    frame = pd.read_csv(path)
    if frame.shape[1] != 1:
        raise ValueError(f"Expected one theta column in {path}, got {frame.shape[1]}.")
    return frame.iloc[:, 0].to_numpy(dtype=float)


def _validate_response_matrix(response_matrix, expected_items, name):
    if response_matrix.ndim != 2 or response_matrix.shape[1] != expected_items:
        raise ValueError(
            f"{name} must have {expected_items} item columns; "
            f"got shape {response_matrix.shape}."
        )
    if not np.isin(response_matrix, [0, 1]).all():
        raise ValueError(f"{name} contains values other than 0 and 1.")


def load_real_data(cfg):
    if cfg.dataset not in DATASET_DIRS:
        raise ValueError(
            f"Unsupported dataset: {cfg.dataset!r}. "
            f"Choose one of {sorted(DATASET_DIRS)}."
        )
    data_dir = DATASET_DIRS[cfg.dataset]

    item_bank = pd.read_csv(data_dir / "real item bank.csv")[["a", "b", "c"]].to_numpy(
        dtype=float
    )
    train_valid_resp = pd.read_csv(
        data_dir / "real responses for training.csv"
    ).to_numpy(dtype=np.int64)
    train_valid_theta = _read_theta(data_dir / "true theta for training.csv")
    test_resp = pd.read_csv(data_dir / "real responses for testing.csv").to_numpy(
        dtype=np.int64
    )
    test_theta = _read_theta(data_dir / "true theta for testing.csv")

    if cfg.n_items > 0:
        if cfg.n_items > len(item_bank):
            raise ValueError("n_items exceeds the available item count.")
        item_bank = item_bank[: cfg.n_items]
        train_valid_resp = train_valid_resp[:, : cfg.n_items]
        test_resp = test_resp[:, : cfg.n_items]

    n_items = len(item_bank)
    _validate_response_matrix(train_valid_resp, n_items, "training responses")
    _validate_response_matrix(test_resp, n_items, "testing responses")
    if len(train_valid_resp) != len(train_valid_theta):
        raise ValueError("Training response/theta row counts do not match.")
    if len(test_resp) != len(test_theta):
        raise ValueError("Testing response/theta row counts do not match.")
    if not 0 < cfg.validation_size < len(train_valid_resp):
        raise ValueError("validation_size must be between 1 and training rows - 1.")
    if cfg.test_length > n_items:
        raise ValueError("test_length cannot exceed the available item count.")
    if cfg.test_length <= 6:
        raise ValueError("test_length must exceed 6 for checkpoint selection.")

    split_rng = np.random.default_rng(cfg.seed)
    indices = split_rng.permutation(len(train_valid_resp))
    valid_indices = indices[: cfg.validation_size]
    train_indices = indices[cfg.validation_size :]
    if cfg.training_size > 0:
        if cfg.training_size > len(train_indices):
            raise ValueError(
                "training_size exceeds rows remaining after validation split."
            )
        train_indices = train_indices[: cfg.training_size]

    if cfg.testing_size > 0:
        if cfg.testing_size > len(test_resp):
            raise ValueError("testing_size exceeds the available testing rows.")
        test_resp = test_resp[: cfg.testing_size]
        test_theta = test_theta[: cfg.testing_size]

    return RealData(
        dataset=cfg.dataset,
        item_bank=item_bank,
        train_resp=train_valid_resp[train_indices],
        train_theta=train_valid_theta[train_indices],
        valid_resp=train_valid_resp[valid_indices],
        valid_theta=train_valid_theta[valid_indices],
        test_resp=test_resp,
        test_theta=test_theta,
    )

In [42]:
def TRAIN(cfg, data):
    get_estimation_method(cfg)
    best_valid = None
    best_state = None

    loss_func = nn.MSELoss()
    eval_net.train()
    optimizer = optim.Adam(eval_net.parameters(), lr=cfg.learning_rate)

    memory_width = cfg.input_size * 2 + 3 + action_space
    memory = np.zeros((cfg.memory_capacity, memory_width))
    memory_counter = 0
    learn_step_counter = 0

    for examinee in range(len(data.train_resp)):
        state = np.array([np.random.rand() - 0.5])
        item_id = np.empty(0, dtype=np.int64)
        resp = np.empty(0, dtype=np.int64)

        for step in range(cfg.test_length):
            action = Choose_Action(item_id, state, cfg.epsilon)
            reward = FI(data.item_bank[action], data.train_theta[examinee])
            item_id = np.concatenate((item_id, action))
            resp = np.concatenate((resp, data.train_resp[examinee, action]))

            theta_hat = estimate_theta_single(
                cfg, data.item_bank, data.item_bank[item_id], resp, state[-1]
            )
            next_state = np.array([theta_hat])
            terminal = float(step == cfg.test_length - 1)
            next_available = np.ones(action_space, dtype=np.float32)
            next_available[item_id] = 0.0

            memory[memory_counter % cfg.memory_capacity, :] = np.hstack(
                (state, action, reward, next_state, terminal, next_available)
            )
            memory_counter += 1
            state = next_state

            if memory_counter >= cfg.batch_size:
                sample_size = min(memory_counter, cfg.memory_capacity)
                batch_memory = memory[np.random.choice(sample_size, cfg.batch_size), :]
                next_state_start = cfg.input_size + 2
                next_state_end = next_state_start + cfg.input_size
                terminal_col = next_state_end
                next_available_start = terminal_col + 1

                batch_state = torch.FloatTensor(batch_memory[:, : cfg.input_size]).to(
                    device
                )
                batch_action = torch.LongTensor(
                    batch_memory[:, cfg.input_size : cfg.input_size + 1].astype(int)
                ).to(device)
                batch_reward = torch.FloatTensor(
                    batch_memory[:, cfg.input_size + 1 : cfg.input_size + 2]
                ).to(device)
                batch_next_state = torch.FloatTensor(
                    batch_memory[:, next_state_start:next_state_end]
                ).to(device)
                batch_terminal = torch.FloatTensor(
                    batch_memory[:, terminal_col : terminal_col + 1]
                ).to(device)
                batch_next_available = torch.BoolTensor(
                    batch_memory[:, next_available_start:].astype(bool)
                ).to(device)

                q_eval = eval_net(batch_state).gather(1, batch_action)
                q_next = target_net(batch_next_state).detach()
                q_next = q_next.masked_fill(~batch_next_available, -torch.inf)
                q_next_max = q_next.max(1)[0].view(cfg.batch_size, 1)
                q_next_max = q_next_max.masked_fill(batch_terminal.bool(), 0.0)
                q_target = batch_reward + cfg.gamma * q_next_max
                loss = loss_func(q_eval, q_target)

                optimizer.zero_grad()
                loss.backward()
                Apply_Positive_Constraint(eval_net)
                optimizer.step()

                learn_step_counter += 1
                if learn_step_counter % cfg.q_network_iteration == 0:
                    target_net.load_state_dict(eval_net.state_dict())

        if (examinee + 1) % cfg.validation_interval == 0:
            eval_net.eval()
            valid_bias = np.zeros((cfg.test_length, len(data.valid_resp)))
            state = np.expand_dims(np.random.rand(len(data.valid_resp)) - 0.5, axis=0)
            item_id = np.empty((0, len(data.valid_resp)), dtype=np.int64)
            resp = np.empty((0, len(data.valid_resp)), dtype=np.int64)

            for step in range(cfg.test_length):
                action = Choose_Action_Test(item_id, state)
                step_resp = data.valid_resp[np.arange(len(data.valid_resp)), action]
                item_id = np.concatenate((item_id, action[np.newaxis, :]))
                resp = np.concatenate((resp, step_resp[np.newaxis, :]))
                theta_hat = estimate_theta_batch(
                    cfg, data.item_bank, item_id, resp, state[-1]
                )
                state = theta_hat[np.newaxis, :]
                valid_bias[step] = theta_hat - data.valid_theta

            step_valid = np.column_stack(
                (
                    np.arange(1, cfg.test_length + 1),
                    np.mean(valid_bias, axis=1),
                    np.sqrt(np.mean(valid_bias**2, axis=1)),
                    np.mean(np.abs(valid_bias), axis=1),
                )
            )
            print(f"subject: {examinee + 1}\n\n{step_valid}\n")
            result_valid = np.mean(step_valid[6:, 1:], axis=0)
            if best_valid is None or result_valid[1] < best_valid[1]:
                best_valid = result_valid
                best_state = copy.deepcopy(eval_net.state_dict())
            eval_net.train()

    return best_state

In [43]:
def TEST(cfg, data):
    get_estimation_method(cfg)
    np.random.seed(cfg.seed)

    with torch.no_grad():
        eval_net.eval()
        testing_size = len(data.test_resp)
        state = np.expand_dims(np.random.rand(testing_size) - 0.5, axis=0)
        item_id = np.empty((0, testing_size), dtype=np.int64)
        resp = np.empty((0, testing_size), dtype=np.int64)
        theta_history = np.empty((0, testing_size), dtype=float)
        summary_rows = []

        for step in range(cfg.test_length):
            action = Choose_Action_Test(item_id, state)
            step_resp = data.test_resp[np.arange(testing_size), action]
            item_id = np.concatenate((item_id, action[np.newaxis, :]))
            resp = np.concatenate((resp, step_resp[np.newaxis, :]))
            theta_hat = estimate_theta_batch(
                cfg, data.item_bank, item_id, resp, state[-1]
            )
            theta_history = np.concatenate((theta_history, theta_hat[np.newaxis, :]))

            bias = theta_hat - data.test_theta
            row = {
                "step": step + 1,
                "Bias": np.mean(bias),
                "RMSE": np.sqrt(np.mean(bias**2)),
                "MAE": np.mean(np.abs(bias)),
            }
            summary_rows.append(row)
            print(
                "step {step:g}, bias {Bias:.3f}, rmse {RMSE:.3f}, mae {MAE:.3f}".format(
                    **row
                )
            )
            state = theta_hat[np.newaxis, :]

        records = pd.DataFrame(
            {
                "userID": np.repeat(np.arange(1, testing_size + 1), cfg.test_length),
                "step": np.tile(np.arange(1, cfg.test_length + 1), testing_size),
                "itemID": (item_id + 1).T.reshape(-1),
                "resp": resp.T.reshape(-1),
                "theta_est": theta_history.T.reshape(-1),
                "bias": (theta_history - data.test_theta).T.reshape(-1),
            }
        )
        summary = pd.DataFrame(summary_rows)

        RESULTS_DIR.mkdir(parents=True, exist_ok=True)
        stem = f"real_{data.dataset}_DQN_{estimation_label(cfg)}_gamma_{cfg.gamma}"
        records_path = RESULTS_DIR / f"records_{stem}.csv"
        summary_path = RESULTS_DIR / f"summary_{stem}.csv"
        records.to_csv(records_path, index=False)
        summary.to_csv(summary_path, index=False)
        print(f"\nSaved records to: {records_path}")
        print(f"Saved summary to: {summary_path}")
        return records, summary

In [44]:
cfg = Config(
    # Network
    input_size=1,
    first_hidden=50,
    second_hidden=30,
    dropout_rate=0.0,
    # Training
    test_length=40,
    gamma=0.5,
    memory_capacity=1000,
    epsilon=0.1,
    batch_size=128,
    q_network_iteration=40,
    learning_rate=1e-3,
    training_size=0,
    validation_size=200,
    validation_interval=50,
    # Ability estimation: 'MLE' | 'EAP'
    estimation_method="MLE",
    n_quad=62,
    prior_low=-4.0,
    prior_high=4.0,
    # Real response data
    dataset="LNIRT_CredentialForm1",
    n_items=0,
    testing_size=0,
    seed=20260430,
)

data = load_real_data(cfg)
item_bank = data.item_bank
action_space = len(item_bank)

print(f"dataset     : {data.dataset}")
print(f"item bank   : {data.item_bank.shape}")
print(f"training    : {data.train_resp.shape}")
print(f"validation  : {data.valid_resp.shape}")
print(f"testing     : {data.test_resp.shape}")
print(f"Config      : {cfg}")

dataset     : LNIRT_CredentialForm1
item bank   : (170, 3)
training    : (945, 170)
validation  : (200, 170)
testing     : (491, 170)
Config      : Config(input_size=1, first_hidden=50, second_hidden=30, dropout_rate=0.0, test_length=40, gamma=0.5, memory_capacity=1000, epsilon=0.1, batch_size=128, q_network_iteration=40, learning_rate=0.001, training_size=0, validation_size=200, validation_interval=50, estimation_method='MLE', n_quad=62, prior_low=-4.0, prior_high=4.0, dataset='LNIRT_CredentialForm1', n_items=0, testing_size=0, seed=20260430)


In [45]:
set_global_seed(cfg.seed)
eval_net = Net(
    cfg.input_size,
    cfg.first_hidden,
    cfg.second_hidden,
    action_space,
    cfg.dropout_rate,
).to(device)
target_net = Net(
    cfg.input_size,
    cfg.first_hidden,
    cfg.second_hidden,
    action_space,
    cfg.dropout_rate,
).to(device)
eval_net.initialize()
target_net.initialize()

best_state = TRAIN(cfg, data)
assert best_state is not None, (
    "No checkpoint was saved. Increase training_size or lower validation_interval."
)
eval_net.load_state_dict(best_state)

MODEL_DIR.mkdir(parents=True, exist_ok=True)
model_path = (
    MODEL_DIR / f"dqn_real_{data.dataset}_{estimation_label(cfg)}_gamma_{cfg.gamma}.pt"
)
torch.save(eval_net.state_dict(), model_path)
print(f"Model saved to: {model_path}")

records, summary = TEST(cfg, data)

subject: 50

[[ 1.          0.79946973  1.85629397  1.63528563]
 [ 2.          0.48807738  1.63841822  1.33848335]
 [ 3.          0.51025754  1.6237193   1.29660744]
 [ 4.          0.45123295  1.5674662   1.2330014 ]
 [ 5.          0.21104472  1.24890448  0.94570601]
 [ 6.          0.21421794  1.1914181   0.89650719]
 [ 7.          0.13894233  0.99689728  0.74614402]
 [ 8.          0.12453411  0.89988984  0.68832823]
 [ 9.          0.11904456  0.88402798  0.6664338 ]
 [10.          0.11928933  0.88894784  0.67047118]
 [11.          0.0892964   0.87113271  0.645055  ]
 [12.          0.09889244  0.8545128   0.61330968]
 [13.          0.10364421  0.83587639  0.59113887]
 [14.          0.0932762   0.79919724  0.58120475]
 [15.          0.10731588  0.76558464  0.56601717]
 [16.          0.09449154  0.77726162  0.56611444]
 [17.          0.10281678  0.76295327  0.56403901]
 [18.          0.10296537  0.7423431   0.53829797]
 [19.          0.11930339  0.74401349  0.53862703]
 [20.          0.1